In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori,association_rules
from mlxtend.preprocessing import TransactionEncoder
import matplotlib.pyplot as plt
import networkx as nx
print("MARKET BASKET ANALYSIS")
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
transactions=[
    ['Bread',"Milk",'Eggs'],
    ['Bread',"Butter","Jam"],
    ["Milk","Bread","Butter"],
    ['Bread',"Milk","Butter","Eggs"],
    ["Milk","Eggs","Cereal"],
    ["Bread","Eggs","Milk"],
    ["Butter","Jam","Bread"],
    ["Bread","Milk","Butter"],
    ["Milk","Cereal","Eggs"],
    ["Bread","Butter","Jam","Milk"]
]
print(len(transactions))
print("Sample Transactions:")
for i in range(3):
  print(f"Transactions {i+1}:{transactions[i]}")


In [ ]:
te=TransactionEncoder()
te_arr=te.fit(transactions).transform(transactions)
df=pd.DataFrame(te_arr,columns=te.columns_)
print("Binary matrix (first 5 rows)")
print(df.head())

In [ ]:
print("STATISTICS")
item=df.sum().sort_values(ascending=False)
print("Item frequencies")
print(item)
trans=[len(t) for t in transactions]
print(f"Average Items: {np.mean(trans):.2f}")
print(f"Min Items: {np.min(trans)}")
print(f"Max Items: {np.max(trans)}")

In [ ]:
print("APRIORI RESULTS")
support=0.3
itemsets=apriori(df,min_support=support,use_colnames=True)
print(f"Frequent Itemsets (support>={support}):")
print(f"Total: {len(itemsets)}")
print(itemsets)

In [ ]:
rules=association_rules(itemsets,metric='lift',min_threshold=1.0)
print(f"Association rules generated {len(rules)}")
print("All Rules")
print(rules[['antecedents','consequents','support','confidence','lift']])

In [ ]:
rules=rules[rules['confidence']>=0.5]
print(f"Rules with confidence >=0.5: {len(rules)}")

In [ ]:
print("TOP RULES")
print("TOP 5 RULES BY LIFT")
top_lift=rules.nlargest(5,'lift')
for i,(idx,row) in enumerate(top_lift.iterrows(),1):
  ante=','.join(list(row['antecedents']))
  cons=",".join(list(row['consequents']))
  print(f"{i}.{ante}->{cons}")
  print(f"Support: {row['support']:.3f}")
  print(f"Confidence: {row['confidence']:.2f}")
  print(f"Lift: {row['lift']:.2f}")

In [ ]:
print("TOP 5 RULES BY CONFIDENCE:")
top_conf=rules.nlargest(5,'confidence')
for i,(idx,row) in enumerate(top_conf.iterrows(),1):
  ante=",".join(list(row['antecedents']))
  cons=",".join(list(row['consequents']))
  print(f"{i} {ante}->{cons}")
  print(f"Support: {row['support']:.2f},Confidence: {row['confidence']:.2f}")
  print(f"Lift: {row['lift']:.2f}")

In [ ]:
plt.figure(figsize=(10,6))
plt.bar(range(len(item)),item.values)
plt.xticks(range(len(item)),item.index,rotation=45)
plt.title('Item Frequencies')
plt.xlabel("Items")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
if len(rules)>0:
  plt.figure(figsize=(8,6))
  g=nx.DiGraph()
  for idx,row in top_lift.iterrows():
    ante=",".join(list(row['antecedents']))
    cons=','.join(list(row['consequents']))
    g.add_edge(ante,cons,weight=row['lift'])
    pos=nx.spring_layout(g)
    nx.draw(g,pos,with_labels=True,node_color='lightblue',node_size=2000
            ,font_size=10,font_weight="bold",arrows=True,arrowstyle="->",
            arrowsize=20)
    plt.title("Top Rules Network")
    plt.show()

In [ ]:

print("BUSINESS INTERPRETATION")
print("What the rules mean:")
print("If a customer buys [Item A],they will likely to also buy [item B]")
print("Lift> 1 means items are positively associated")
print("Confidence show how often the rule holds true")
print("Key Finding:")
if len(rules)>0:
  best=rules.nlargest(1,'lift').iloc[0]
  ante=','.join(best['antecedents'])
  cons=','.join(best['consequents'])
  print(f'Strongest Associations: {ante}->{cons}')
  print(f"Lift: {best['lift']:.2f}")
print("Markerting Suggestions")
print("1. Store Layout: Place associated items near each other")
print("2. Promotions: Offer discounts on pairs that are frequently bought together")
print("3. Cross-selling: Train staff to suggest complementary items")
print("4. Product placement: Create themed displays (e.g. breakfast section)")

rules[['antecedents','consequents','support','confidence','lift']].to_csv('simple_rules.csv',index=False)
print("Results saved to simple_rules.csv")

In [ ]:
import pandas as pd
import time
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Part A: Data Loading (Grocery Dataset)
dataset = [['Milk', 'Bread', 'Butter'],
           ['Bread', 'Butter'],
           ['Milk', 'Bread'],
           ['Milk', 'Butter', 'Bread', 'Eggs'],
           ['Bread', 'Butter', 'Eggs']]

# Preprocessing into binary matrix [cite: 609]
te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_ary, columns=te.columns_)

# Part B: FP-Growth vs Apriori Timing
start = time.time()
frequent_itemsets_apriori = apriori(df, min_support=0.4, use_colnames=True)
apriori_time = time.time() - start

start = time.time()
frequent_itemsets_fp = fpgrowth(df, min_support=0.4, use_colnames=True)
fpgrowth_time = time.time() - start

# Part C: Rule Generation [cite: 614, 615]
rules = association_rules(frequent_itemsets_fp, metric="lift", min_threshold=1.0)

print(f"Apriori Time: {apriori_time:.5f}s")
print(f"FP-Growth Time: {fpgrowth_time:.5f}s")
print("\nTop Association Rules:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head())